# 04 · M1.1 — each asset type gets its own transformation

## Executive summary (read this first)

M1 (notebook 03) used log returns for every level series and lost to M0 (1.110), entirely because of interest rates: a
yield moves in percentage points, not in percent. **M1.1 picks the transformation per asset type**, following the
target definitions:

| asset type | assets | target | step used | forecast at h | vs M0 |
|---|---|---|---|---|---|
| rates | UST_2Y, 5Y, 7Y, 10Y, 30Y | level | Δy = y_t − y_{t−1} | y + Σ Δy | same as M0 |
| **FX** | AUD, BRL, CAD, CHF, CNY, DKK, EUR, GBP, INR, JPY, NOK, NZD, SEK | level | **r = log(S_t ÷ S_{t−1})** | **S × exp(Σ r)** | **changed** |
| **CPI** | CPI_ALL (monthly) | level | **g = log(CPI_t ÷ CPI_{t−1})** | **CPI × exp(Σ g)** | **changed** |
| payrolls | NFP (monthly) | level | ΔNFP (total employment, checked in §1) | NFP + Σ ΔNFP | same as M0 |
| unemployment | UNRATE (monthly) | level | ΔU in percentage points | U + Σ ΔU | same as M0 |
| equity factors | MKT, HML, MOM, SMB, QMJ | log_return | the daily log return itself | Σ r (cumulative) | same as M0 |

The table lives in **`transformations.toml`** (this folder): every notebook reads a unit's `card.toml`, looks up each
asset there and applies the rule (`unit_rules` in §1). Everything else is M0: the last 300 rows, the gap rule, the drift (mean step), the covariance of the steps, one path
across horizons, normal shocks, the same seed and 500 draws. A unit with several assets uses each asset's own step, with
the covariance between them as in M0. **Release rule:** on a monthly series, if no new release falls before the target
date (zero monthly steps ahead), the forecast is the last released value. (On the released units every monthly target is
at least two releases ahead, so the rule never fires here; it is in the code for hidden units.)

**The test:** notebook 01's backtest dates and answers; the competition score per unit = `Σ weight × mean(M1.1's category
score) ÷ mean(M0's)`. Below 1, M1.1 beats M0. Normal shocks only: Student-t is the next step.

Results: §5 (per unit), §6 (per family and asset type), §8 (in short). No realized outcome is read.

## 0 — Setup: notebook 01's backtest dates, answers and M0 scores

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())

def load_unit(u):
    """Card fields and each asset's history up to the as-of, read with M0's rule (first panel holding the asset)."""
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof = card["task"]["id"], card["provenance"]["data_cutoff"]
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": t.get("target_type", "level"), "asof": asof, "rows": rows,
            "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
DATES = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    DATES[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {len(M0_SCORES):,} backtest dates (from notebook 01)")

103 units, 23,635 backtest dates (from notebook 01)


## 1 — The transformation per asset (from `transformations.toml`), and two checks from the target definitions

`unit_rules(unit_dir)` reads the unit's `card.toml` (asset ids, target type) and returns `{asset: transform}` with
`transform` one of `diff` (x_t − x_{t−1}, M0's step), `log_diff` (log(x_t ÷ x_{t−1})) or `as_return` (the row already is
a log return). Rules: a log-return target is `as_return`; a listed asset takes its type's transform; an unlisted asset
takes `diff`.

- **NFP:** is the column total payroll employment (difference it) or jobs added each month (do not)? Total employment is
  around 130–160 million, i.e. 130,000–160,000 in thousands; jobs added is a few hundred.
- **Release rule:** how many monthly steps ahead is each monthly target?

In [2]:
RULES = tomllib.loads((HERE / "transformations.toml").read_text())

def asset_type(a):
    return RULES["assets"].get(a, "other")

def transform_for(asset, target_type):
    """card.toml's target type + the asset -> 'diff' / 'log_diff' / 'as_return' (the rules in transformations.toml)."""
    if target_type == "log_return":
        return RULES["default"]["log_return"]
    t = RULES["assets"].get(asset)
    return RULES["types"][t]["transform"] if t else RULES["default"]["level"]

def unit_rules(unit_dir):
    """Read a unit's card.toml and return {asset: transform}."""
    t = tomllib.loads((pathlib.Path(unit_dir) / "card.toml").read_text())["targets"]
    return {a: transform_for(a, t.get("target_type", "level")) for a in t["asset_ids"]}

for u, U in UNITS_DATA.items():
    U["rules"] = unit_rules(UNITS / u)

assets = sorted({a for U in UNITS_DATA.values() for a in U["assets"]})
display(pd.DataFrame({"asset type": [asset_type(a) for a in assets],
                      "transform": [transform_for(a, "log_return" if asset_type(a) == "equity_factor" else "level") for a in assets]},
                     index=pd.Index(assets, name="asset")).sort_values("asset type").T)
print("unit_rules on a few units:")
for u in ["t2-F1-greater-confidence-2024", "t2-F2-eur-parity-2022", "t2-F3-boj-ust-channel-2023",
          "t2-F1-cpi-glidepath-2023", "t2-F4-covid-nfp-2020", "t2-F4-short-vol-2018"]:
    print(f"  {u:<30} {UNITS_DATA[u]['rules']}")

nfp = [U for U in UNITS_DATA.values() if "NFP" in U["assets"]][0]
v = np.array([x for _, x in nfp["rows"]["NFP"]])
print(f"NFP ({nfp['unit']}): last 3 values {v[-3:].tolist()} (value_unit: {nfp['card']['targets'].get('value_unit')})")
print(f"  -> about {v[-1] / 1000:,.0f} million jobs: TOTAL payroll employment, so the step is the monthly change "
      f"(last: {v[-1] - v[-2]:+,.0f} thousand), and it is not differenced twice.")
print("\nmonthly units, steps ahead per horizon (the release rule fires only at 0):")
for U in UNITS_DATA.values():
    if U["id"] in m0._MONTHLY_STEPS:
        print(f"  {U['unit']:<28} {U['assets']}  horizons {U['horizons']} -> monthly steps {list(U['steps'].values())}")

asset,CPI_ALL,HML,MKT,MOM,QMJ,SMB,AUD,BRL,CAD,CHF,CNY,DKK,EUR,GBP,INR,JPY,NOK,NZD,SEK,NFP,UST_10Y,UST_2Y,UST_30Y,UST_5Y,UST_7Y,UNRATE
asset type,cpi,equity_factor,equity_factor,equity_factor,equity_factor,equity_factor,fx,fx,fx,fx,fx,fx,fx,fx,fx,fx,fx,fx,fx,payrolls,rates,rates,rates,rates,rates,unemployment
transform,log_diff,as_return,as_return,as_return,as_return,as_return,log_diff,log_diff,log_diff,log_diff,log_diff,log_diff,log_diff,log_diff,log_diff,log_diff,log_diff,log_diff,log_diff,diff,diff,diff,diff,diff,diff,diff


unit_rules on a few units:
  t2-F1-greater-confidence-2024  {'UST_2Y': 'diff'}
  t2-F2-eur-parity-2022          {'EUR': 'log_diff'}
  t2-F3-boj-ust-channel-2023     {'UST_10Y': 'diff', 'JPY': 'log_diff'}
  t2-F1-cpi-glidepath-2023       {'CPI_ALL': 'log_diff'}
  t2-F4-covid-nfp-2020           {'NFP': 'diff'}
  t2-F4-short-vol-2018           {'MKT': 'as_return'}
NFP (t2-F4-covid-nfp-2020): last 3 values [151998.0, 152271.0, 152544.0] (value_unit: thousands of jobs (PAYEMS, as-published first-release vintage))
  -> about 153 million jobs: TOTAL payroll employment, so the step is the monthly change (last: +273 thousand), and it is not differenced twice.

monthly units, steps ahead per horizon (the release rule fires only at 0):
  t2-F1-cpi-glidepath-2023     ['CPI_ALL']  horizons [140, 160] -> monthly steps [8, 9]
  t2-F1-sahm-watch-2024        ['UNRATE']  horizons [145, 165] -> monthly steps [8, 9]
  t2-F4-covid-nfp-2020         ['NFP']  horizons [21] -> monthly steps [2]
  t2-F4-cpi-vin

## 2 — M1.1 at any date

M0's code (notebook 01's `m0_draws`) with one switch per asset: an FX or CPI series takes its steps on `log(value)`,
which `m0._steps` turns into log changes with the same gap rule, and its draws go back through `last × exp(·)`. Every
other asset is exactly M0. Check: every unit without an FX or CPI asset equals M0 exactly.

In [3]:
def m11_draws(U, origin):
    """M1.1's 500 draws at `origin`, shape (500, n_assets, n_horizons), card order."""
    assets, horizons, tt = U["assets"], U["horizons"], U["target_type"]
    by_asset = sorted(assets)
    steps, last, log_mode = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - m0._WINDOW): k]
        log_mode[a] = U["rules"][a] == "log_diff" and min(x for _, x in rows) > 0
        if log_mode[a]:                                          # FX, CPI: log changes
            steps[a], _ = m0._steps([(d, np.log(x)) for d, x in rows], "level")
            last[a] = rows[-1][1]
        else:                                                    # rates, NFP, UNRATE (changes); factors (returns): M0
            steps[a], lv = m0._steps(rows, tt)
            last[a] = lv if tt == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]
    s = U["steps"]
    mean = np.array([(0.0 if log_mode[a] else last[a]) + s[h] * mu[ai[a]] for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10
    jittered = cov + 1e-9 * np.eye(len(cells))
    try:
        factor = np.linalg.cholesky(jittered)
    except np.linalg.LinAlgError:
        factor = np.diag(np.sqrt(np.diag(jittered)))
    rng = np.random.default_rng(zlib.crc32(U["id"].encode()) & 0x7FFFFFFF)      # M0's seed: the same random numbers
    samples = mean + rng.standard_normal((m0.N_DRAWS, len(cells))) @ factor.T
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(samples[:, ci]) if log_mode[a] else samples[:, ci]
        if s[h] <= 0:                                            # release rule: no new release before the target
            col = np.full(m0.N_DRAWS, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

changed = {u for u, U in UNITS_DATA.items() if "log_diff" in U["rules"].values()}
same = [u for u in UNIT_IDS if u not in changed]
d = max(float(np.abs(m11_draws(UNITS_DATA[u], UNITS_DATA[u]["asof"]) - m0.draws(UNITS / u)).max()) for u in same)
print(f"{len(same)} units have no FX or CPI asset: max |M1.1 - M0| at the real as-of = {d:.1e}  ->  identical to M0")
print(f"{len(changed)} units differ from M0 (an FX or CPI asset)")
assert d == 0

U = UNITS_DATA["t2-F2-eur-parity-2022"]
both = {"M0": m0.draws(UNITS / U["unit"])[:, 0, 0], "M1.1": m11_draws(U, U["asof"])[:, 0, 0]}
q = [0.01, 0.05, 0.5, 0.95, 0.99]
print(f"\nan FX example, {U['unit']} ({U['assets'][0]}, h = {U['horizons'][0]}, last = {U['rows'][U['assets'][0]][-1][1]}):")
display(pd.DataFrame({m: np.quantile(v, q) for m, v in both.items()}, index=[f"{int(p * 100)}%" for p in q]).T)

61 units have no FX or CPI asset: max |M1.1 - M0| at the real as-of = 0.0e+00  ->  identical to M0
42 units differ from M0 (an FX or CPI asset)

an FX example, t2-F2-eur-parity-2022 (EUR, h = 64, last = 1.0469):


,1%,5%,50%,95%,99%
M0,0.9235,0.9462,1.013,1.075,1.109
M1.1,0.9376,0.9568,1.016,1.074,1.107


## 3 — Score M1.1 at every backtest date (the same dates and answers as notebook 01)

In [4]:
def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)

def score_unit(u):
    U = UNITS_DATA[u]
    return [{"unit": u, "family": U["family"], "origin": o, **parts(U["card"], m11_draws(U, o).reshape(m0.N_DRAWS, -1), y)}
            for o, y in DATES[u]]

t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    M11_SCORES = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
assert len(M11_SCORES) == len(M0_SCORES)
print(f"scored M1.1 on {len(M11_SCORES):,} dates in {time.time() - t0:.0f}s")

scored M1.1 on 23,635 dates in 1s


## 4 — The competition score per unit

`score = Σ w_k × mean(category_k of M1.1) ÷ mean(category_k of M0)`, over the unit's backtest dates. M0 = 1.0.

In [5]:
cats = {"marginal": "CRPS", "joint": "variogram", "tail": "tail"}
mean1 = M11_SCORES.groupby("unit")[["marginal", "joint", "tail", "composite"]].mean()
mean0 = M0_SCORES.groupby("unit")[["marginal", "joint", "tail", "composite"]].mean()
rows = []
for u in UNIT_IDS:
    U = UNITS_DATA[u]; w = m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
    types = sorted({asset_type(a) for a in U["assets"]})
    r = {"family": U["family"], "unit": u, "asset type": types[0] if len(types) == 1 else " + ".join(types),
         "differs from M0": u in changed, "dates": int((M11_SCORES.unit == u).sum())}
    for (k, name), wk in zip(cats.items(), w):
        r[f"M1.1 {name}"] = mean1.loc[u, k]
        r[f"{name} ÷ M0"] = mean1.loc[u, k] / mean0.loc[u, k] if wk else np.nan
    r["M1.1 weighted sum"], r["M0 weighted sum"] = mean1.loc[u, "composite"], mean0.loc[u, "composite"]
    r["score vs M0"] = sum(wk * mean1.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, w) if wk)
    rows.append(r)
RESULT = pd.DataFrame(rows).set_index(["family", "unit"])

## 5 — Per unit (the units where M1.1 differs from M0; the rest score exactly 1.0)

In [6]:
RESULT[RESULT["differs from M0"]].drop(columns="differs from M0").sort_values("score vs M0")

asset type  dates  M1.1 CRPS  CRPS ÷ M0  M1.1 variogram  variogram ÷ M0  M1.1 tail  tail ÷ M0  M1.1 weighted sum  M0 weighted sum  score vs M0
family unit                                                                                                                                                                         
T2-F1  t2-F1-cpi-glidepath-2023              cpi     11      4.307     0.9035          0.3211          0.8157      1.231     0.7765              2.496            2.819       0.8517
T2-F4  t2-F4-cpi-vintage-2022                cpi     10     0.9905     0.9499               0             NaN     0.1839     0.7528               0.76           0.8146       0.8936
T2-F1  t2-F1-eur-range-2017                   fx    250    0.06852     0.9623         0.01476          0.8999    0.01065      1.008            0.04082          0.04264       0.9528
T2-F2  t2-F2-fragile-five-brl-2013            fx    117     0.2396      1.007               0             NaN    0.03421      0.861             0.1809           0.1812       0.9656
       t2-F2-abenomics-2012                   fx    186      2.231     0.9714               0             NaN     0.2901      0.951              1.676            1.727       0.9656
T2-F3  t2-F3-safe-haven-paradox-2011  fx + rates    164     0.9326     0.9839           2.567          0.9503     0.1121     0.9974              1.259            1.307       0.9765
T2-F2  t2-F2-opec-cad-2014                    fx    216    0.01954     0.9878               0             NaN   0.002417     0.9823            0.01465          0.01483       0.9862
       t2-F2-boj-thin-signal-2014             fx    214       2.62      1.009               0             NaN     0.3666     0.9423              1.976            1.966       0.9898
T2-F3  t2-F3-dollar-vortex-2022       fx + rates    330     0.6046     0.9958           2.915          0.9927    0.08701     0.9772              1.194            1.202       0.9911
       t2-F3-boj-ust-channel-2023     fx + rates    344      1.323     0.9998            1.74          0.9878     0.1963     0.9767              1.223             1.23       0.9916
T2-F2  t2-F2-nirp-paradox-2016                fx    233      3.061      1.016               0             NaN     0.4386     0.9348              2.312            2.285       0.9931
       t2-F2-whatever-it-takes-2012           fx    180    0.05353     0.9937               0             NaN   0.005074     0.9961            0.03969          0.03993       0.9944
T2-F4  t2-F4-jpy-crowding-2024                fx    363      1.603      1.002               0             NaN     0.2288      0.978               1.21            1.209       0.9952
T2-F1  t2-F1-aud-on-hold-2016                 fx    237    0.05322     0.9879         0.01416          0.9953   0.006057      1.015            0.03207           0.0324       0.9955
T2-F4  t2-F4-nok-covid-2020                   fx    297     0.1341      1.002               0             NaN    0.01756     0.9792             0.1008           0.1007       0.9956
T2-F2  t2-F2-eur-parity-2022                  fx    329    0.02588     0.9985               0             NaN     0.0031     0.9894            0.01937          0.01941       0.9959
T2-F3  t2-F3-scandies-stress-2022             fx    333      0.158      1.001          0.7319          0.9921    0.02028     0.9881             0.3026           0.3043        0.996
T2-F2  t2-F2-shunto-nirp-exit-2024            fx    355      2.986      1.004               0             NaN     0.4819     0.9783              2.271            2.266       0.9966
       t2-F2-trichet-alertness-2008           fx    119    0.02571     0.9956               0             NaN   0.003088     0.9996            0.01925          0.01933       0.9967
T2-F4  t2-F4-aud-gfc-2008                     fx    125    0.01615      1.002               0             NaN   0.002365      0.986            0.01221           0.0122       0.9974
       t2-F4-eur-warnings-2022                fx    326 

## 6 — Per family and per asset type

The mean competition score over all units (as on the leaderboard), and over the units where M1.1 differs.

In [7]:
def summary(g):
    c = g[g["differs from M0"]]
    return {"units": len(g), "score vs M0 (mean, all)": g["score vs M0"].mean(), "units that differ": len(c),
            "those: mean": c["score vs M0"].mean() if len(c) else np.nan,
            "those: better": f'{(c["score vs M0"] < 1).sum()}/{len(c)}',
            "CRPS ÷ M0 (geo)": geo(c["CRPS ÷ M0"]) if len(c) else np.nan,
            "tail ÷ M0 (geo)": geo(c["tail ÷ M0"]) if len(c) else np.nan}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*RESULT.groupby("family"), ("all", RESULT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in RESULT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)

per family:


,units,"score vs M0 (mean, all)",units that differ,those: mean,those: better,CRPS ÷ M0 (geo),tail ÷ M0 (geo)
family,,,,,,,
T2-F1,23,0.9955,7,0.9853,3/7,0.9936,0.9682
T2-F2,27,0.999,15,0.9981,10/15,1.003,0.9845
T2-F3,22,0.9993,11,0.9986,7/11,1.004,0.9767
T2-F4,31,0.9968,9,0.9888,5/9,0.9954,0.9682
all,103,0.9976,42,0.9941,25/42,1,0.9762


per asset type:


,units,"score vs M0 (mean, all)",units that differ,those: mean,those: better,CRPS ÷ M0 (geo),tail ÷ M0 (geo)
asset type,,,,,,,
cpi,2,0.8727,2,0.8727,2/2,0.9264,0.7645
equity_factor,16,1,0,NaN,0/0,NaN,NaN
fx,35,1.001,35,1.001,19/35,1.005,0.9895
fx + rates,5,0.9948,5,0.9948,4/5,1,0.9795
payrolls,1,1,0,NaN,0/0,NaN,NaN
rates,43,1,0,NaN,0/0,NaN,NaN
unemployment,1,1,0,NaN,0/0,NaN,NaN


## 7 — Save

In [8]:
M11_SCORES.to_parquet(DATA / "m1_1_backtest_scores.parquet", index=False)
RESULT.to_csv(DATA / "m1_1_vs_m0_per_unit.csv")
FAMILY.to_csv(DATA / "m1_1_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "m1_1_vs_m0_by_asset_type.csv")
for f in ("m1_1_backtest_scores.parquet", "m1_1_vs_m0_per_unit.csv", "m1_1_vs_m0_per_family.csv", "m1_1_vs_m0_by_asset_type.csv"):
    print(f"data/{f:<32} {(DATA / f).stat().st_size / 1e3:8.1f} kB")

data/m1_1_backtest_scores.parquet        867.1 kB
data/m1_1_vs_m0_per_unit.csv              17.9 kB
data/m1_1_vs_m0_per_family.csv             0.6 kB
data/m1_1_vs_m0_by_asset_type.csv          0.5 kB


## 8 — In short

In [9]:
a = FAMILY.loc["all"]
print(f"M1.1 vs M0 on {len(M11_SCORES):,} backtest dates: competition score {a['score vs M0 (mean, all)']:.4f} over all "
      f"{int(a['units'])} units (1.0 = M0, lower is better).")
print(f"M1.1 differs from M0 on {int(a['units that differ'])} units (an FX or CPI asset): mean {a['those: mean']:.4f}, "
      f"better on {a['those: better']}.")
for t, r in BY_TYPE.iterrows():
    if r["units that differ"]:
        print(f"  {t:<14} {int(r['units that differ'])} units: {r['those: mean']:.4f} ({r['those: better']} better)")
for f in [f for f in FAMILY.index if f != "all"]:
    print(f"  {f}: {FAMILY.loc[f, 'score vs M0 (mean, all)']:.4f} over all its units")

M1.1 vs M0 on 23,635 backtest dates: competition score 0.9976 over all 103 units (1.0 = M0, lower is better).
M1.1 differs from M0 on 42 units (an FX or CPI asset): mean 0.9941, better on 25/42.
  cpi            2 units: 0.8727 (2/2 better)
  fx             35 units: 1.0010 (19/35 better)
  fx + rates     5 units: 0.9948 (4/5 better)
  T2-F1: 0.9955 over all its units
  T2-F2: 0.9990 over all its units
  T2-F3: 0.9993 over all its units
  T2-F4: 0.9968 over all its units
